# Phase 1 — Clean and build panel

## To-do

- [x] Load current register + filter Skilled Worker
- [x] Write `clean_company_name()` and create `company_key`
- [x] Load all snapshots, add `snapshot_date`, stack into one dataframe
- [x] Deduplicate within each snapshot; document counts
- [x] Build panel + `first_seen`, `last_seen`, `still_active`, `tenure_days`
- [x] Show how many entered / exited between two dates
- [x] Save panel + company summary parquet to `data/processed/`

**Done when:** we can answer how many sponsors entered and exited between snapshot A and B.

**Result (2024-01-05 → 2025-01-08):** entered 27,940 · exited 2,929 · stayed 78,634

In [5]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
project_root = cwd if (cwd / "data" / "raw").exists() else cwd.parent

csv_path = project_root / "data" / "raw" / "sponsor_register_2026-07-28.csv"
df = pd.read_csv(csv_path)

print(df.shape)
print(df["Route"].value_counts().head())

skilled = df[df["Route"] == "Skilled Worker"].copy()
print("Skilled Worker rows:", len(skilled))
skilled.head()

(142635, 5)
Route
Skilled Worker                                           122698
Global Business Mobility: Senior or Specialist Worker     10419
Tier 2 Ministers of Religion                               1932
Creative Worker                                            1585
Charity Worker                                             1532
Name: count, dtype: int64
Skilled Worker rows: 122698


,Organisation Name,Town/City,County,Type & Rating,Route
0,ALT,Glasgow,NaN,Worker (A rating),Skilled Worker
1,Asian African Foods Ltd,London,NaN,Worker (A rating),Skilled Worker
2,BOLTWHIZ LIMITED,Dunfermline,Scotland,Worker (A rating),Skilled Worker
3,Bossmans Retail Abergavenny Ltd,Abergavenny,NaN,Worker (A rating),Skilled Worker
4,BRANOS OXFORD LTD T/A LILO,Oxford,NaN,Worker (A rating),Skilled Worker


In [6]:
name = "ACME Software Limited"
name = name.lower().strip()
print(name)

acme software limited


In [12]:
import re

def clean_company_name(name: str) -> str:
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = re.sub(r"[^a-z0-9\s]", " ", name)  # remove punctuation
    name = re.sub(r"\s+", " ", name).strip()
    for suffix in [" limited", " ltd", " plc", " llp"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)].strip()
    return name

In [14]:
examples = [
    "ACME Software Limited",
    "Tesco PLC",
    "Foo Bar LLP",
    "  Hello World Ltd. ",
]
for e in examples:
    print(e, "->", clean_company_name(e))

ACME Software Limited -> acme software
Tesco PLC -> tesco
Foo Bar LLP -> foo bar
  Hello World Ltd.  -> hello world ltd


In [15]:
skilled["company_key"] = skilled["Organisation Name"].map(clean_company_name)
skilled[["Organisation Name", "company_key"]].head(20)

,Organisation Name,company_key
0,ALT,alt
1,Asian African Foods Ltd,asian african foods
2,BOLTWHIZ LIMITED,boltwhiz
3,Bossmans Retail Abergavenny Ltd,bossmans retail abergavenny
4,BRANOS OXFORD LTD T/A LILO,branos oxford ltd t a lilo
5,BRITANNIA BUSINESS CONSULTING LIMITED,britannia business consulting
6,Brooke Healthcare Ltd,brooke healthcare
7,BW Refrigeration & Air Conditioning Limited,bw refrigeration air conditioning
8,BZK PIZZAWORKS LTD,bzk pizzaworks
9,C. BECHSTEIN HALL LIMITED,c bechstein hall


## Load all snapshots

Find every `sponsor_register_*.csv`, tag with `snapshot_date`, keep Skilled Worker only, clean names.

In [ ]:
import sys

sys.path.insert(0, str(project_root / "src"))
from clean_names import clean_company_name

files = sorted((project_root / "data" / "raw").rglob("sponsor_register_*.csv"))
print(len(files), "files found:")
for f in files:
    print(" ", f.relative_to(project_root))

frames = []
for path in files:
    # date is in the filename: sponsor_register_YYYY-MM-DD.csv
    snapshot_date = pd.to_datetime(path.stem.replace("sponsor_register_", ""))
    part = pd.read_csv(path)
    part = part[part["Route"] == "Skilled Worker"].copy()
    part["snapshot_date"] = snapshot_date
    part["company_key"] = part["Organisation Name"].map(clean_company_name)
    frames.append(part)
    print(snapshot_date.date(), "->", len(part), "Skilled Worker rows")

all_skilled = pd.concat(frames, ignore_index=True)
print("\nStacked rows:", len(all_skilled))
print("Dates:", sorted(all_skilled["snapshot_date"].dt.date.unique()))
all_skilled.head()

## Deduplicate within each snapshot

Same `company_key` on the same date = keep one row.

In [ ]:
before = len(all_skilled)
# drop empty keys and exact company+date duplicates
panel = all_skilled[all_skilled["company_key"] != ""].copy()
panel = panel.drop_duplicates(subset=["snapshot_date", "company_key"], keep="first")
after = len(panel)

print("Rows before dedupe:", before)
print("Rows after dedupe:", after)
print("Duplicates removed:", before - after)
print("Unique companies:", panel["company_key"].nunique())
print("\nRows per snapshot after dedupe:")
print(panel.groupby("snapshot_date").size())

## Company summary

One row per company: first seen, last seen, still active, tenure.

In [ ]:
latest_date = panel["snapshot_date"].max()

company_summary = (
    panel.groupby("company_key", as_index=False)
    .agg(
        first_seen=("snapshot_date", "min"),
        last_seen=("snapshot_date", "max"),
        n_snapshots=("snapshot_date", "nunique"),
        example_name=("Organisation Name", "first"),
        town=("Town/City", "first"),
    )
)
company_summary["still_active"] = company_summary["last_seen"] == latest_date
company_summary["tenure_days"] = (
    company_summary["last_seen"] - company_summary["first_seen"]
).dt.days

print("Companies:", len(company_summary))
print("Still active:", int(company_summary["still_active"].sum()))
print("Latest snapshot:", latest_date.date())
company_summary.head()

## Entry / exit between two dates

Done-when test: how many joined and left between date A and date B.

In [ ]:
date_a = pd.Timestamp("2024-01-05")
date_b = pd.Timestamp("2025-01-08")

set_a = set(panel.loc[panel["snapshot_date"] == date_a, "company_key"])
set_b = set(panel.loc[panel["snapshot_date"] == date_b, "company_key"])

entered = set_b - set_a
exited = set_a - set_b
stayed = set_a & set_b

print(f"Between {date_a.date()} and {date_b.date()}:")
print(f"  On A: {len(set_a)}")
print(f"  On B: {len(set_b)}")
print(f"  Entered: {len(entered)}")
print(f"  Exited:  {len(exited)}")
print(f"  Stayed:  {len(stayed)}")

## Save clean outputs

In [ ]:
out_dir = project_root / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

panel_path = out_dir / "sponsor_panel.parquet"
summary_path = out_dir / "sponsor_company_summary.parquet"

panel.to_parquet(panel_path, index=False)
company_summary.to_parquet(summary_path, index=False)

print("Saved:", panel_path)
print("Saved:", summary_path)
print("Panel rows:", len(panel))
print("Summary rows:", len(company_summary))